# NeuroAgent: A Board of AI Doctors for EEG Review

**High-Risk Project – AI in Healthcare**

## About this Project

This project explores whether multiple AI models can review the same EEG and provide a better explanation than a single AI model.

In hospitals, doctors often discuss difficult cases with other specialists before making a decision. This project follows a similar idea. Three AI "doctor" agents review the same EEG data. Each agent gives its own opinion. A moderator agent compares the answers, points out where they agree or disagree, and provides a final summary.

This project is for research only. It is not intended to replace doctors or make medical decisions.

## Dataset

This project uses the **CHB-MIT Scalp EEG Database**, a public dataset of EEG recordings from children with epilepsy.

The dataset contains seizure and non-seizure recordings but does not include seizure type labels. Because of this, this project focuses on describing EEG findings instead of identifying specific seizure types.

Future work may use datasets such as **TUSZ**, which include seizure type information.

## Code

GitHub / Colab Link:
*Add link before submission.*

## 1. Setup

We install the packages we need:
- `openai` to call the AI models
- `mne` to read EEG `.edf` files
- `pandas` / `numpy` for data handling


In [34]:
!pip install openai mne pandas numpy scikit-learn --quiet


In [35]:
import os
import json
import time
import numpy as np
import pandas as pd
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
gv_client = OpenAI()

# Three doctor agents, each on a different model.
gv_model_neurologist = "gpt-4o"
gv_model_epileptologist = "o3-mini"
gv_model_neurophysiologist = "gpt-4o-mini"


## 2. Download the EEG Data (CHB-MIT)

For this project, we download only a small number of EEG files instead of the full 42 GB dataset.

The selected files include:
- A few EEG recordings that contain seizures.
- A few EEG recordings that do not contain seizures.

Using both types of recordings helps the AI agents compare normal and seizure activity.



In [36]:
gv_base_url = "https://physionet.org/files/chbmit/1.0.0"

# case -> list of files (seizure files + one non-seizure file + the summary text file)
gv_cases = {
    "chb01": ["chb01_03.edf", "chb01_04.edf", "chb01_01.edf", "chb01-summary.txt"],
    "chb05": ["chb05_06.edf", "chb05_13.edf", "chb05_01.edf", "chb05-summary.txt"],
    "chb08": ["chb08_02.edf", "chb08_05.edf", "chb08_01.edf", "chb08-summary.txt"],
}

for lv_case, lv_files in gv_cases.items():
    lv_dir = f"/content/chbmit/{lv_case}"
    os.makedirs(lv_dir, exist_ok=True)
    for lv_fname in lv_files:
        lv_url = f"{gv_base_url}/{lv_case}/{lv_fname}"
        !wget -q -nc -P {lv_dir} {lv_url}
    print(f"{lv_case}: downloaded {len(lv_files)} files")


chb01: downloaded 4 files
chb05: downloaded 4 files
chb08: downloaded 4 files


## 3. Prepare the EEG Data

Raw EEG data is a long list of numbers. Before an AI model can understand it, we need to convert it into a simple summary.

First, we read the `-summary.txt` files to find the exact start and end times of each seizure.

Next, we select:
- A seizure segment.
- A non-seizure segment from the same patient.

Finally, we calculate the power in the main brain wave bands:
- Delta
- Theta
- Alpha
- Beta
- Gamma

These features provide a simple summary of the EEG and are used as input for the AI agents.

In [37]:
import mne
import re

def parse_summary_seizures(lv_summary_path):
    """Read a chbXX-summary.txt file and return a list of
    (filename, seizure_start_seconds, seizure_end_seconds) for every seizure listed."""
    lv_text = open(lv_summary_path).read()
    lv_blocks = lv_text.split("File Name:")[1:]
    lv_results = []
    for lv_block in lv_blocks:
        lv_fname = lv_block.strip().split()[0]
        lv_starts = re.findall(r"Seizure Start Time:\s*(\d+)", lv_block)
        lv_ends = re.findall(r"Seizure End Time:\s*(\d+)", lv_block)
        for lv_s, lv_e in zip(lv_starts, lv_ends):
            lv_results.append((lv_fname, int(lv_s), int(lv_e)))
    return lv_results


def band_power_features(lv_data, lv_sfreq):
    """Compute simple frequency-band power features for a window of EEG data.
    lv_data shape: (n_channels, n_samples). Returns a dict of band -> average power."""
    lv_psds, lv_freqs = mne.time_frequency.psd_array_welch(
        lv_data, sfreq=lv_sfreq, fmin=0.5, fmax=45, n_fft=min(256, lv_data.shape[1]), verbose=False
    )
    lv_bands = {"delta": (0.5, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 45)}
    lv_out = {}
    for lv_name, (lv_lo, lv_hi) in lv_bands.items():
        lv_mask = (lv_freqs >= lv_lo) & (lv_freqs < lv_hi)
        lv_out[lv_name] = float(lv_psds[:, lv_mask].mean())
    return lv_out


def extract_case_windows(lv_case, lv_files, lv_window_sec=20):
    """For a given case, extract one seizure window and one non-seizure window
    per seizure file, plus one window from the dedicated non-seizure file."""
    lv_dir = f"/content/chbmit/{lv_case}"
    lv_summary_path = f"{lv_dir}/{lv_case}-summary.txt"
    lv_seizures = parse_summary_seizures(lv_summary_path)
    lv_records = []

    for lv_fname, lv_start, lv_end in lv_seizures:
        lv_path = f"{lv_dir}/{lv_fname}"
        if not os.path.exists(lv_path):
            continue
        lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
        lv_sfreq = lv_raw.info["sfreq"]

        # Seizure window
        lv_seg = lv_raw.copy().crop(tmin=lv_start, tmax=min(lv_start + lv_window_sec, lv_end))
        lv_feat = band_power_features(lv_seg.get_data(), lv_sfreq)
        lv_records.append({"case": lv_case, "file": lv_fname, "label": "seizure", **lv_feat})

        # Non-seizure window from the same file, well before the seizure starts
        if lv_start > lv_window_sec + 30:
            lv_seg2 = lv_raw.copy().crop(tmin=10, tmax=10 + lv_window_sec)
            lv_feat2 = band_power_features(lv_seg2.get_data(), lv_sfreq)
            lv_records.append({"case": lv_case, "file": lv_fname, "label": "non_seizure", **lv_feat2})

    return lv_records


gv_all_records = []
for lv_case, lv_files in gv_cases.items():
    gv_all_records.extend(extract_case_windows(lv_case, lv_files))

gv_features_df = pd.DataFrame(gv_all_records)
print(f"Extracted {len(gv_features_df)} EEG windows (seizure + non-seizure)")
gv_features_df


/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)
/tmp/ipykernel_2288/2228378103.py:45: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  lv_raw = mne.io.read_raw_edf(lv_path, preload=True, verbose=False)


Extracted 8 EEG windows (seizure + non-seizure)


,case,file,label,delta,theta,alpha,beta,gamma
0,chb01,chb01_03.edf,seizure,1.900132e-09,4.280013e-10,4.331971e-11,1.241013e-11,2.033422e-11
1,chb01,chb01_03.edf,non_seizure,2.721512e-10,5.793261e-11,3.712411e-11,4.240661e-12,6.575679e-13
2,chb01,chb01_04.edf,seizure,2.619219e-09,4.451644e-10,4.349532e-11,1.901407e-11,2.399497e-11
3,chb01,chb01_04.edf,non_seizure,1.452702e-10,4.246701e-11,1.027651e-11,3.691532e-12,2.308902e-12
4,chb05,chb05_06.edf,seizure,5.647274e-10,1.316556e-09,2.042092e-10,4.533573e-11,8.222060e-12
5,chb05,chb05_06.edf,non_seizure,1.092846e-09,4.888177e-10,2.948911e-11,1.020008e-11,6.988870e-12
6,chb05,chb05_13.edf,seizure,7.658479e-10,4.980844e-09,2.049728e-09,3.625813e-10,1.041426e-10
7,chb05,chb05_13.edf,non_seizure,6.530586e-10,1.197691e-10,3.083264e-11,3.448828e-11,4.683773e-11


### Convert EEG Features to Text

The AI agents work better with text than with raw numbers. We convert each set of EEG features into a short, easy-to-read description of the EEG window.


In [38]:
def features_to_text(lv_row):
    return (
        f"EEG window from case {lv_row['case']}, file {lv_row['file']}.\n"
        f"Average band power (relative strength of brain-wave frequencies):\n"
        f"  Delta (0.5-4 Hz): {lv_row['delta']:.3g}\n"
        f"  Theta (4-8 Hz): {lv_row['theta']:.3g}\n"
        f"  Alpha (8-13 Hz): {lv_row['alpha']:.3g}\n"
        f"  Beta (13-30 Hz): {lv_row['beta']:.3g}\n"
        f"  Gamma (30-45 Hz): {lv_row['gamma']:.3g}\n"
    )

gv_features_df["eeg_description"] = gv_features_df.apply(features_to_text, axis=1)
print(gv_features_df["eeg_description"].iloc[0])


EEG window from case chb01, file chb01_03.edf.
Average band power (relative strength of brain-wave frequencies):
  Delta (0.5-4 Hz): 1.9e-09
  Theta (4-8 Hz): 4.28e-10
  Alpha (8-13 Hz): 4.33e-11
  Beta (13-30 Hz): 1.24e-11
  Gamma (30-45 Hz): 2.03e-11



## 4. Use Medical References

Doctors use medical knowledge when reviewing EEGs. This project gives the AI agents a small collection of epilepsy reference notes.

For each EEG description, the program finds the most relevant notes and shares them with the AI agents. This helps the agents base their answers on medical information instead of only the EEG features.

In [39]:
gv_literature = {
    "focal_seizure": (
        "Focal seizures typically show localized rhythmic discharges, often with increased "
        "beta or gamma activity in specific channels, reflecting a seizure onset restricted "
        "to one brain region before possibly spreading."
    ),
    "generalized_seizure": (
        "Generalized seizures typically show synchronous, widespread discharges across most "
        "or all channels simultaneously, often with prominent slow-wave (delta/theta) activity."
    ),
    "normal_background": (
        "Normal background EEG in an awake or lightly drowsy pediatric patient shows a mix of "
        "alpha and beta activity without sustained high-amplitude rhythmic discharges."
    ),
    "ictal_pattern": (
        "The ictal (seizure) period is often characterized by a sudden change in frequency and "
        "amplitude compared to the pre-seizure baseline, most clearly seen as a shift toward "
        "higher beta/gamma power or rhythmic delta/theta buildup."
    ),
}


def extract_band_powers(lv_description):
    lv_bands = {}
    for lv_match in re.finditer(r"(\w+) \([\d.]+-[\d.]+ Hz\): ([\d.eE+-]+)", lv_description):
        lv_bands[lv_match.group(1).lower()] = float(lv_match.group(2))
    return lv_bands


def retrieve_literature(lv_eeg_description, lv_top_k=2):
    lv_bands = extract_band_powers(lv_eeg_description)
    lv_dominant_band = max(lv_bands, key=lv_bands.get) if lv_bands else ""
    lv_query_words = set(re.findall(r"[a-z]+", lv_eeg_description.lower()))
    lv_query_words |= {lv_dominant_band, lv_dominant_band}  # extra weight for the dominant band

    lv_scored = []
    for lv_key, lv_text in gv_literature.items():
        lv_snippet_words = re.findall(r"[a-z]+", lv_text.lower())
        lv_overlap = sum(1 for lv_w in lv_snippet_words if lv_w in lv_query_words)
        lv_scored.append((lv_overlap, lv_key, lv_text))

    lv_scored.sort(key=lambda lv_x: lv_x[0], reverse=True)
    lv_top = lv_scored[:lv_top_k]
    return "\n".join(f"[{lv_key}] {lv_text}" for _, lv_key, lv_text in lv_top)


## 5. The AI Doctor Agents

This project uses three AI agents. Each one reviews the same EEG information from a different point of view.

To make the opinions different, each agent uses a different AI model and has a different clinical role.

| Agent | Model | Role |
|---|---|---|
| Doctor 1 | GPT-4o | General neurologist |
| Doctor 2 | o3-mini | Epilepsy specialist |
| Doctor 3 | GPT-4o-mini | EEG specialist |

In [40]:
gv_role_prompts = {
    "neurologist": (
        "You are a general neurologist reviewing a pediatric EEG window. Give your best-guess "
        "differential (is this seizure activity or not, and if so, does it look more focal or "
        "generalized?), your reasoning, and how confident you are (low/medium/high)."
    ),
    "epileptologist": (
        "You are an epileptologist, a neurologist who subspecializes in epilepsy and handles "
        "ambiguous or hard-to-classify cases. Review this pediatric EEG window carefully, "
        "consider more than one possible interpretation, and explain the reasoning behind each. "
        "State your confidence (low/medium/high) for your leading interpretation."
    ),
    "neurophysiologist": (
        "You are a clinical neurophysiologist who focuses on the raw EEG signal itself rather "
        "than the patient's clinical presentation. Focus your reasoning on the frequency-band "
        "pattern described below: what it suggests about the underlying brain activity, and how "
        "confident you are (low/medium/high)."
    ),
}

def call_llm(lv_model, lv_system_prompt, lv_user_prompt, lv_retries=2):
    """Call an OpenAI model and return the response text.
    Retries a couple of times if the call fails, instead of crashing the whole run."""
    for lv_attempt in range(lv_retries + 1):
        try:
            lv_resp = gv_client.chat.completions.create(
                model=lv_model,
                messages=[
                    {"role": "system", "content": lv_system_prompt},
                    {"role": "user", "content": lv_user_prompt},
                ],
            )
            return lv_resp.choices[0].message.content
        except Exception as lv_e:
            if lv_attempt == lv_retries:
                print("Failed after retries:", lv_e)
                return f"[ERROR: {lv_e}]"
            time.sleep(2)


def doctor_agent(lv_role, lv_model, lv_eeg_description, lv_literature):
    """One doctor agent: combines a role, a model, the EEG description, and grounding literature."""
    lv_system_prompt = gv_role_prompts[lv_role]
    lv_user_prompt = (
        f"EEG window description:\n{lv_eeg_description}\n\n"
        f"Relevant reference literature:\n{lv_literature}\n\n"
        "Give your assessment, your reasoning, and your confidence level."
    )
    return call_llm(lv_model, lv_system_prompt, lv_user_prompt)


## 6. Test the AI Doctor Team

Now let's test the system with one EEG example and see what each AI doctor says.

In [41]:
gv_example_row = gv_features_df.iloc[0]
gv_example_description = gv_example_row["eeg_description"]
gv_example_literature = retrieve_literature(gv_example_description)

gv_doctor_outputs = {}
gv_doctor_outputs["neurologist"] = doctor_agent("neurologist", gv_model_neurologist, gv_example_description, gv_example_literature)
gv_doctor_outputs["epileptologist"] = doctor_agent("epileptologist", gv_model_epileptologist, gv_example_description, gv_example_literature)
gv_doctor_outputs["neurophysiologist"] = doctor_agent("neurophysiologist", gv_model_neurophysiologist, gv_example_description, gv_example_literature)

for lv_role, lv_output in gv_doctor_outputs.items():
    print(f"===== {lv_role.upper()} =====")
    print(lv_output)
    print()


===== NEUROLOGIST =====
Based on the provided EEG window description, this does not appear to be seizure activity. Here's the reasoning:

1. **Dominant Frequencies:** The dominant frequency bands in this EEG window are Delta (0.5-4 Hz) with a power of 1.9e-09 and Theta (4-8 Hz) with a power of 4.28e-10. These lower frequencies are not typically associated with clear ictal patterns, which often show a sudden shift towards higher beta or gamma power or rhythmic delta/theta buildup. Although there is some theta activity present, it is not described as "rhythmic" or significantly higher than normal baseline which would suggest an ictal pattern.

2. **Low Alpha and Beta Activity:** The Alpha (8-13 Hz) and Beta (13-30 Hz) activity levels are very low compared to Delta and Theta. In a typical awake pediatric EEG, we would expect more prominent alpha activity, especially posteriorly if the patient is awake. These low values could suggest the patient is in a more relaxed or early drowsy state, 

## 7. The Moderator Agent

The moderator reviews the opinions from all three AI doctors. It summarizes where they agree, where they disagree, and gives a final summary of the case.

In [42]:
gv_model_moderator = "gpt-4o"

def moderator_agent(lv_doctor_outputs):
    lv_system_prompt = (
        "You are a moderator summarizing a discussion between three doctors (a neurologist, "
        "an epileptologist, and a neurophysiologist) who each reviewed the same EEG window "
        "independently. Identify where they agree, where they disagree, and explain possible "
        "reasons for any disagreement. Do not simply pick a winner -- the goal is to make the "
        "uncertainty and reasoning visible, the way a real multidisciplinary discussion would."
    )
    lv_user_prompt = "\n\n".join(
        f"{lv_role.upper()} said:\n{lv_text}" for lv_role, lv_text in lv_doctor_outputs.items()
    )
    return call_llm(gv_model_moderator, lv_system_prompt, lv_user_prompt)

gv_moderator_summary = moderator_agent(gv_doctor_outputs)
print(gv_moderator_summary)


The discussion among the neurologist, epileptologist, and neurophysiologist regarding the same EEG window reveals both agreements and disagreements in interpretation and confidence levels. 

**Areas of Agreement:**
1. **Dominance of Delta Waves:** All experts agree that delta waves are the predominant activity in the EEG window, which is often associated with non-ictal states, such as deep sleep or structural brain abnormalities in certain awake states.
   
2. **Lack of Beta/Gamma Activity:** Each specialist mentions the minimal power in the beta and gamma frequencies, which are typically seen in ictal events or active cognitive states.

3. **Non-Ictal Interpretation:** Both the neurologist and epileptologist favor a non-ictal state interpretation, leaning towards states like sleep or other non-seizure alterations of consciousness.

4. **Medium Confidence Level:** All experts share a medium confidence level due to the lack of direct clinical context, emphasizing the need for a holistic

## 8. Analyze All EEG Samples

Now we run the same process on every EEG sample. The three AI doctors review each sample, and the moderator summarizes their opinions.

The results are saved in a table for analysis and discussion in the report.

In [43]:
gv_results = []

for lv_idx, lv_row in gv_features_df.iterrows():
    lv_description = lv_row["eeg_description"]
    lv_literature = retrieve_literature(lv_description)

    lv_outputs = {
        "neurologist": doctor_agent("neurologist", gv_model_neurologist, lv_description, lv_literature),
        "epileptologist": doctor_agent("epileptologist", gv_model_epileptologist, lv_description, lv_literature),
        "neurophysiologist": doctor_agent("neurophysiologist", gv_model_neurophysiologist, lv_description, lv_literature),
    }
    lv_moderator_summary = moderator_agent(lv_outputs)

    gv_results.append({
        "case": lv_row["case"],
        "file": lv_row["file"],
        "true_label": lv_row["label"],  # seizure vs non_seizure, from the dataset annotation
        "neurologist_opinion": lv_outputs["neurologist"],
        "epileptologist_opinion": lv_outputs["epileptologist"],
        "neurophysiologist_opinion": lv_outputs["neurophysiologist"],
        "moderator_summary": lv_moderator_summary,
    })
    print(f"Processed {lv_row['case']} / {lv_row['file']} ({lv_row['label']})")

pd.set_option("display.max_colwidth", None)
gv_results_df = pd.DataFrame(gv_results)
gv_results_df


Processed chb01 / chb01_03.edf (seizure)
Processed chb01 / chb01_03.edf (non_seizure)
Processed chb01 / chb01_04.edf (seizure)
Processed chb01 / chb01_04.edf (non_seizure)
Processed chb05 / chb05_06.edf (seizure)
Processed chb05 / chb05_06.edf (non_seizure)
Processed chb05 / chb05_13.edf (seizure)
Processed chb05 / chb05_13.edf (non_seizure)


,case,file,true_label,neurologist_opinion,epileptologist_opinion,neurophysiologist_opinion,moderator_summary
0,chb01,chb01_03.edf,seizure,"Based on the given EEG window data and relevant literature, I will provide an analysis regarding whether this represents seizure activity or not.\n\n**Assessment**:\n1. **Delta Power (0.5-4 Hz):** The power in this band is relatively high at 1.9e-09, which is more prominent compared to other frequencies.\n2. **Theta Power (4-8 Hz):** This is also somewhat elevated at 4.28e-10.\n3. **Alpha Power (8-13 Hz):** The alpha power is significantly lower at 4.33e-11.\n4. **Beta Power (13-30 Hz):** Similarly, this power is low at 1.24e-11.\n5. **Gamma Power (30-45 Hz):** The gamma power is relatively low at 2.03e-11.\n\nGiven the data, the predominant activity is in the lower frequency delta and theta bands. In the context of seizure activity, an ""ictal period is often characterized by a sudden change in frequency and amplitude compared to the pre-seizure baseline, most clearly seen as a shift toward higher beta/gamma power or rhythmic delta/theta buildup."" The current data does not mention a specific baseline, but from the information available, there seems to be some prominence in the delta and theta ranges, which could indicate rhythmic slow activity.\n\n**Reasoning**:\n- If this EEG were part of an ictal episode, one might expect more prominence in higher frequency bands (beta/gamma) or very high amplitude rhythmic delta/theta, which might suggest a seizure pattern.\n- The predominance of delta and theta activity could suggest some kind of pathologic slowing, but without specifics on high amplitude or rhythm, it's challenging to definitively call it a seizure.\n- Given the suppression of alpha and beta activity, and the lack of increased beta/gamma power, this window may not clearly represent seizure activity. However, a subtle focal seizure might reflect just changes in slower frequencies.\n\n**Confidence Level**: Medium\n\nWithout more context or visual inspection, there is some uncertainty, as EEG interpretation often includes assessment in conjunction with clinical correlation. This pattern might suggest background slowing or a non-specific abnormality rather than categorical ictal activity. However, pathological significance, such as a focal seizure with rhythmic slowing, cannot entirely be ruled out. More context or visual clues could adjust this differential diagnosis.","Below is one way to think through the findings:\n\n1. The data show that the delta band (0.5–4 Hz) power is much higher than the other bands. Theta power is next in magnitude but is still considerably lower than delta, and the contributions of alpha, beta, and gamma are very small in comparison. In a normal awake or lightly drowsy pediatric EEG one would expect a reasonable mix of alpha and beta frequencies, rather than a delta‐dominant spectrum.\n\n2. One interpretation is that this window might represent ictal (seizure) activity. Some seizures, rather than “revving up” the high‐frequency (beta/gamma) bands, may present with a buildup of slow (delta/theta) rhythmic activity. The literature reference “[ictal_pattern]” supports that an ictal period can be signified by a rhythmic buildup in delta/theta. Hence, the large delta power seen here could reflect a rhythmic delta ictal discharge, especially in an epilepsy patient such as those in the CHB dataset. I assign a medium level of confidence to this interpretation because while the quantitative shift toward delta fits an ictal pattern in some cases, the full diagnosis of ictal activity usually requires both frequency analysis and a review of the waveform morphology and clinical context.\n\n3. An alternative explanation is that the recording represents a state different from typical awake background – for example, a sleep stage or a state of encephalopathy that shows slowing. Deep sleep (slow-wave sleep) is characterized by high delta activity, but in such a state one migh

### Save the results




In [44]:
gv_results_df.to_csv("/content/neuroagent_v1_results.csv", index=False)
print("Saved results to /content/neuroagent_v1_results.csv")


Saved results to /content/neuroagent_v1_results.csv


## 9. Compare with a Simple Machine Learning Model

Before using the AI doctor system, we compare it with a simple machine learning model.

We use **logistic regression** with the same EEG features (delta, theta, alpha, beta, and gamma band power) to classify seizure and non-seizure samples.

Because this project uses a small dataset, we evaluate the model using **leave-one-out cross-validation**, where each EEG sample is tested once while the remaining samples are used for training.

This provides a simple baseline for comparison with the AI doctor system.

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut
from sklearn.preprocessing import StandardScaler

gv_feature_cols = ["delta", "theta", "alpha", "beta", "gamma"]
gv_X = gv_features_df[gv_feature_cols].values
gv_y = (gv_features_df["label"] == "seizure").astype(int).values

gv_loo = LeaveOneOut()
gv_baseline_predictions = []

for lv_train_idx, lv_test_idx in gv_loo.split(gv_X):
    lv_scaler = StandardScaler()
    lv_X_train = lv_scaler.fit_transform(gv_X[lv_train_idx])
    lv_X_test = lv_scaler.transform(gv_X[lv_test_idx])
    lv_clf = LogisticRegression()
    lv_clf.fit(lv_X_train, gv_y[lv_train_idx])
    lv_pred = lv_clf.predict(lv_X_test)[0]
    gv_baseline_predictions.append(lv_pred)

gv_features_df["baseline_prediction"] = [
    "seizure" if lv_p == 1 else "non_seizure" for lv_p in gv_baseline_predictions
]
gv_baseline_correct = (gv_features_df["baseline_prediction"] == gv_features_df["label"]).sum()
gv_baseline_accuracy = gv_baseline_correct / len(gv_features_df)

print(f"Baseline logistic regression accuracy (leave-one-out): {gv_baseline_accuracy:.2f} "
      f"({gv_baseline_correct}/{len(gv_features_df)} correct)")
gv_features_df[["case", "file", "label", "baseline_prediction"] + gv_feature_cols]


Baseline logistic regression accuracy (leave-one-out): 0.75 (6/8 correct)


,case,file,label,baseline_prediction,delta,theta,alpha,beta,gamma
0,chb01,chb01_03.edf,seizure,seizure,1.900132e-09,4.280013e-10,4.331971e-11,1.241013e-11,2.033422e-11
1,chb01,chb01_03.edf,non_seizure,non_seizure,2.721512e-10,5.793261e-11,3.712411e-11,4.240661e-12,6.575679e-13
2,chb01,chb01_04.edf,seizure,seizure,2.619219e-09,4.451644e-10,4.349532e-11,1.901407e-11,2.399497e-11
3,chb01,chb01_04.edf,non_seizure,non_seizure,1.452702e-10,4.246701e-11,1.027651e-11,3.691532e-12,2.308902e-12
4,chb05,chb05_06.edf,seizure,non_seizure,5.647274e-10,1.316556e-09,2.042092e-10,4.533573e-11,8.222060e-12
5,chb05,chb05_06.edf,non_seizure,seizure,1.092846e-09,4.888177e-10,2.948911e-11,1.020008e-11,6.988870e-12
6,chb05,chb05_13.edf,seizure,seizure,7.658479e-10,4.980844e-09,2.049728e-09,3.625813e-10,1.041426e-10
7,chb05,chb05_13.edf,non_seizure,non_seizure,6.530586e-10,1.197691e-10,3.083264e-11,3.448828e-11,4.683773e-11


## 10. Limitations

- The CHB-MIT dataset labels **seizure** and **non-seizure** recordings, but it does not include seizure types. Because of this, the project describes EEG patterns instead of predicting specific seizure types.
- The three AI doctors use different OpenAI models and different roles, but they are not completely independent because they come from the same model family.
- The medical reference library is small. A larger collection of medical articles could improve the results.
- The EEG features are simple and use only brain wave band power. More advanced EEG features could improve performance.
- This project uses a small number of EEG samples, so it should be considered a proof of concept.
- The logistic regression model is included only as a simple baseline for comparison.
- In some cases, the AI doctors gave different opinions on the same EEG sample. This shows that simple EEG features may not always provide enough information, and more detailed EEG analysis could improve future versions.

## 11. Future Work

- Use the **TUSZ** dataset to identify different seizure types instead of only seizure and non-seizure cases.
- Add another AI agent to suggest possible epilepsy medications based on medical guidelines.
- Develop an AI agent that helps identify when a patient's medication should be reviewed based on changes over time.
- Include AI models from different providers to make the AI doctors more independent and provide a wider range of opinions.